In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_wine
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler


def safe_silhouette(features, labels, mask=None):
    if mask is not None:
        features, labels = features[mask], labels[mask]
    if len(labels) < 2 or len(set(labels)) < 2:
        return None
    return silhouette_score(features, labels)


def main():
    wine = load_wine()
    x = StandardScaler().fit_transform(wine.data)
    y_true = wine.target

    dbscan_labels = DBSCAN(eps=1.5, min_samples=5).fit_predict(x)
    dbscan_clusters = set(dbscan_labels) - {-1}
    noise_mask = dbscan_labels != -1

    kmeans_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(x)

    print("Истинных сортов:", len(set(y_true)))
    print("DBSCAN: кластеров =", len(dbscan_clusters), ", шум =", int((~noise_mask).sum()))
    print("DBSCAN ARI:", round(adjusted_rand_score(y_true, dbscan_labels), 4))
    print("DBSCAN silhouette (включая шум):", safe_silhouette(x, dbscan_labels))
    if noise_mask.any():
        print("DBSCAN silhouette (без шума):", safe_silhouette(x, dbscan_labels, noise_mask))

    print("\nKMeans: кластеров =", len(set(kmeans_labels)))
    print("KMeans ARI:", round(adjusted_rand_score(y_true, kmeans_labels), 4))
    print("KMeans silhouette:", round(silhouette_score(x, kmeans_labels), 4))
    print(
        "\nДля поиска истинных сортов сравните ARI: он оценивает соответствие "
        "кластеров известным меткам и не зависит от номеров кластеров."
    )


if __name__ == "__main__":
    main()


- При `eps=1.5`, `min_samples=5` DBSCAN объявил все 178 объектов шумом: `ARI=0`, silhouette не определён. KMeans с `k=3` получил `ARI=0.8975` и silhouette `0.2849`; в этом запуске KMeans лучше восстановил сорта.
- KMeans обычно занимает `O(I·k·n·d)` времени, где `I` — число итераций. DBSCAN в среднем около `O(n log n)`, в худшем случае `O(n²)`. Точный silhouette score требует `O(n²·d)` времени.